# Mobileye semantic retrieval — table-v2 / chunk-v3

This notebook is a consumer of the versioned evaluation dataset; it does not embed or reconstruct gold labels. Nested evidence and chunk-ID arrays are loaded directly from JSON. The historical pre-migration results remain recorded in the generated baseline report for comparison, but are not presented as current results.

In [1]:
from pathlib import Path
import json

from src.evaluation.evaluate_retrieval import evaluate, write_json_atomic

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASET = PROJECT_ROOT / 'data/evaluation/mobileye_retrieval_gold_v2.json'
CHUNKS = PROJECT_ROOT / 'data/chunks/MBLY/2025-10-K.chunks.jsonl'
EMBEDDINGS = PROJECT_ROOT / 'data/embeddings/MBLY/2025-10-K.bgebase.embeddings.npz'
MANIFEST = EMBEDDINGS.with_suffix('.manifest.json')
OUTPUT = PROJECT_ROOT / 'data/evaluation/mobileye_bgebase_table_v2_baseline.json'

/home/veselin/Documents/Programiranje/edgar-insight-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = json.loads(DATASET.read_text(encoding='utf-8'))
assert dataset['schema_version'] == 2
assert dataset['record_count'] == 60
assert all(record['review_state']['status'] == 'approved' for record in dataset['records'])

# Arrays remain native arrays; no comma splitting or string reconstruction.
[(record['id'], record['new_relevant_chunk_ids']) for record in dataset['records'][:3]]

[('Q01',
  ['MBLY-2025-CHUNK-000038',
   'MBLY-2025-CHUNK-000040',
   'MBLY-2025-CHUNK-000041']),
 ('Q02', ['MBLY-2025-CHUNK-000006', 'MBLY-2025-CHUNK-000029']),
 ('Q03', ['MBLY-2025-CHUNK-000014', 'MBLY-2025-CHUNK-000016'])]

In [3]:
report = evaluate(
    DATASET, CHUNKS, EMBEDDINGS, MANIFEST,
    model_name='bgebase', device='cpu', batch_size=32, top_k=10,
)
write_json_atomic(OUTPUT, report)
report['metrics']

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5309.62it/s]


{'overall': {'question_count': 60,
  'mean_recall_at_k': 0.6166666666666667,
  'mean_reciprocal_rank_at_k': 0.4446494708994709,
  'hit_rate_at_k': 0.6833333333333333},
 'by_evaluation_set': {'chunk_to_query': {'question_count': 34,
   'mean_recall_at_k': 0.7205882352941176,
   'mean_reciprocal_rank_at_k': 0.48786181139122314,
   'hit_rate_at_k': 0.7352941176470589},
  'query_to_chunk': {'question_count': 26,
   'mean_recall_at_k': 0.4807692307692308,
   'mean_reciprocal_rank_at_k': 0.38814102564102565,
   'hit_rate_at_k': 0.6153846153846154}},
 'by_category': {'financials': {'question_count': 10,
   'mean_recall_at_k': 0.31666666666666665,
   'mean_reciprocal_rank_at_k': 0.3458333333333333,
   'hit_rate_at_k': 0.5},
  'industry_and_business': {'question_count': 5,
   'mean_recall_at_k': 0.4666666666666667,
   'mean_reciprocal_rank_at_k': 0.5,
   'hit_rate_at_k': 0.6},
  'multi_chunk': {'question_count': 8,
   'mean_recall_at_k': 0.5625,
   'mean_reciprocal_rank_at_k': 0.458333333333333

In [4]:
# Inspect every incomplete retrieval before any reranking experiment.
[(row['id'], row['recall_at_k'], row['relevant_chunk_ids'], row['retrieved_chunk_ids'])
 for row in report['regressions_for_review']]

[('Q01',
  0.3333333333333333,
  ['MBLY-2025-CHUNK-000038',
   'MBLY-2025-CHUNK-000040',
   'MBLY-2025-CHUNK-000041'],
  ['MBLY-2025-CHUNK-000020',
   'MBLY-2025-CHUNK-000041',
   'MBLY-2025-CHUNK-000002',
   'MBLY-2025-CHUNK-000201',
   'MBLY-2025-CHUNK-000034',
   'MBLY-2025-CHUNK-000019',
   'MBLY-2025-CHUNK-000014',
   'MBLY-2025-CHUNK-000013',
   'MBLY-2025-CHUNK-000207',
   'MBLY-2025-CHUNK-000016']),
 ('Q02',
  0.0,
  ['MBLY-2025-CHUNK-000006', 'MBLY-2025-CHUNK-000029'],
  ['MBLY-2025-CHUNK-000019',
   'MBLY-2025-CHUNK-000014',
   'MBLY-2025-CHUNK-000016',
   'MBLY-2025-CHUNK-000020',
   'MBLY-2025-CHUNK-000034',
   'MBLY-2025-CHUNK-000018',
   'MBLY-2025-CHUNK-000002',
   'MBLY-2025-CHUNK-000015',
   'MBLY-2025-CHUNK-000013',
   'MBLY-2025-CHUNK-000041']),
 ('Q04',
  0.0,
  ['MBLY-2025-CHUNK-000022',
   'MBLY-2025-CHUNK-000023',
   'MBLY-2025-CHUNK-000024'],
  ['MBLY-2025-CHUNK-000041',
   'MBLY-2025-CHUNK-000019',
   'MBLY-2025-CHUNK-000011',
   'MBLY-2025-CHUNK-000020',
   'M